# Exploration of Watershed Boundary (HUC-8) Dataset

In [ ]:
import geopandas as gpd
import pandas as pd

## Initial Investigation

In [ ]:
gpkg_path = "../../data/raw/wbd/WBD_National_GPKG.gpkg"
wbd_df = gpd.read_file(gpkg_path,layer="WBDHU8")
wbd_df.head()

In [ ]:
wbd_df.info()

In [ ]:
wbd_df.describe()

In [ ]:
# Check for duplicate names
duplicates = wbd_df[wbd_df["name"].duplicated(keep=False)].sort_values("name")
print(f"The HUC8 dataset contains {len(duplicates)} watersheds with the same name")
duplicates[["name", "huc8", "states"]].head(30)

Given that there are multiple watersheds with the same name, joining the dam removals based on name can get messy. Instead, a spatial join will be performed below; dams will be joined to the watersheds based on whether or not their coordinates are within a watershed polygon.

## Join to dam removal dataset

In [ ]:
dam_df = pd.read_csv('../../data/raw/dams/ARDamRemovalList_Figshare_Mar2026.csv')
merged = dam_df.merge(
    wbd_df,
    left_on="HUC8",
    right_on="name",
    how="left"
)

print(f"The joined table has {merged.shape[0]} rows and {merged.shape[1]} columns")
print(f"There are {len(merged['HUC8'].dropna().unique())} unique HUC8s")


In [ ]:
print(dam_df.shape)

In [ ]:
# Convert dams to GeoDataFrame
dam_gdf = gpd.GeoDataFrame(
    dam_df,
    geometry=gpd.points_from_xy(
        dam_df["Longitude"],
        dam_df["Latitude"]
    ),
    crs="EPSG:4326"
)

# Make sure WBD uses the same CRS (coordinate system)
wbd_huc8 = wbd_df.to_crs(dam_gdf.crs)

# Spatial join
merged_df = gpd.sjoin(
    dam_gdf,
    wbd_huc8[["huc8", "name", "geometry"]],
    how="left",
    predicate="within"
)

print(merged_df.shape)

merged_df.head(20)

In [ ]:
print(f"There are {len(merged_df['AR_ID'].dropna().unique())} unique dams") # should be 2325

## Extract HUC4 codes from HUC8 codes in joined dataset to get all relevant HUC4 watersheds

This is done so that only a subset of all streams in the US are downloaded - we do not want to download the entire stream dataset as this would be ginormous

In [ ]:
huc8s = merged_df['huc8'].dropna().unique()
print(f"HUC 8s: {list(huc8s)}")

print("\n\n\n")

huc4s = merged_df['huc8'].str[:4].dropna().unique()
print(f"HUC 4s: {list(huc4s)}")

In [ ]:
print(f"The dam removals are in {len(huc4s)} HUC4 watersheds. There are 245 HUC4 watersheds in the US (https://nas.er.usgs.gov/hucs.aspx)")

## Write hucs to text file to aid stream downloads

In [ ]:
# with open("../../data/processed/huc4_list.txt", "w") as f:
#     f.write("\n".join(huc4s))
# print("Success!")

In [ ]:
# # Confirm file contents are correct
# from pathlib import Path

# huc4_file = Path("../../data/processed/huc4_list.txt")
# huc4s = [
#     x.strip()
#     for x in huc4_file.read_text().splitlines()
#     if x.strip()
# ]

# print(len(huc4s))

Examination of the NHDPlus API reveals that only some HUC-4s are downloadable. Further, only some HUC-8s are downloadable. Thus, we must create a second text file for HUC-8s. Download logic will be handled with the python script.

In [ ]:
# with open("../../data/processed/huc8_list.txt", "w") as f:
#     f.write("\n".join(huc8s))
# print("Success!")

## Now, visualize HUC 8 Watersheds and Dam Removals on a map

In [ ]:
%pip install folium

In [ ]:
import folium
from folium import GeoJson, FeatureGroup
from folium.plugins import MarkerCluster

# Folium requires WGS84 coordinates
print("Beginning folium setup...")

relevant_hucs = merged_df["huc8"].dropna().unique()

watersheds = wbd_huc8[
    wbd_huc8["huc8"].isin(relevant_hucs)
].to_crs("EPSG:4326")

# Simplify watershed boundaries for visualization only
watersheds_map = watersheds.copy()

watersheds_map["geometry"] = (
    watersheds_map.geometry.simplify(
        tolerance=0.001,
        preserve_topology=True
    )
)

print("Watersheds simplified for mapping")

dams = merged_df.to_crs("EPSG:4326")

# Center map on the dam dataset
center = [
    dams.geometry.y.mean(),
    dams.geometry.x.mean()
]

m = folium.Map(
    location=center,
    zoom_start=5,
    tiles="OpenStreetMap"
)

# HUC8 watershed polygons
print("Beginning watersheds...")
huc_layer = FeatureGroup(
    name="HUC8 Watersheds",
    show=True
)

GeoJson(
    watersheds_map[["huc8", "name", "geometry"]],
    style_function=lambda feature: {
        "fillColor": "#4C9F70",
        "color": "#28734A",
        "weight": 1,
        "fillOpacity": 0.12
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["huc8", "name"],
        aliases=["HUC8:", "Watershed:"]
    )
).add_to(huc_layer)

print("Watershed layer constructed")

huc_layer.add_to(m)

# Dam removal markers
print("Beginning dams...")

dam_layer = FeatureGroup(
    name="Dam Removals",
    show=True
)

# Keep only valid point geometries
valid_dams = dams[
    dams.geometry.notna()
    & ~dams.geometry.is_empty
    & dams.geometry.is_valid
].copy()

valid_dams = valid_dams[
    valid_dams.geometry.x.between(-180, 180)
    & valid_dams.geometry.y.between(-90, 90)
]

# Add dam locations as one GeoJSON layer
GeoJson(
    valid_dams[["huc8", "name", "geometry"]],
    name="Dam Removals",
    marker=folium.CircleMarker(
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.7
    ),
    tooltip=folium.GeoJsonTooltip(
        fields=["huc8", "name"],
        aliases=["HUC8:", "Watershed:"]
    )
).add_to(dam_layer)

dam_layer.add_to(m)

print("Dam layer constructed")

folium.LayerControl(collapsed=False).add_to(m)

m